In [9]:
import pandas as pd
import ast

# ── 1. LOAD CLEANED-UP ALIAS TABLE ───────────────────────────
alias_df = pd.read_csv("Data Inggris - Data_Inggris_transformed.csv")   # story_id | person | aliases | sentence_id
def safe_parse_alias(val):
    # kalau sudah list, langsung pakai
    if isinstance(val, list):
        return val

    # kalau kosong / NaN
    if pd.isna(val):
        return []

    # ubah ke string dan rapikan
    val = str(val).strip()

    if not val or val.lower() == "nan":
        return []

    # coba parse kalau formatnya memang list/string python
    try:
        parsed = ast.literal_eval(val)
        if isinstance(parsed, list):
            return parsed
        elif isinstance(parsed, str):
            return [parsed]
        else:
            return [str(parsed)]
    except:
        # fallback:
        # kalau isinya dipisah koma, pecah manual
        if "," in val:
            return [x.strip() for x in val.split(",") if x.strip()]
        else:
            return [val]
alias_df["AliasCharacters"] = alias_df["AliasCharacters"].apply(safe_parse_alias)

def split_ids(val):
    if pd.isna(val) or str(val).strip() == "":
        return []
    return [int(x) for x in str(val).split(",") if str(x).strip().isdigit()]

alias_df["sentenceID"] = alias_df["sentenceID"].apply(split_ids)

# ── 2. LOAD WORD-LEVEL TOKEN TABLE & BUILD FULL SENTENCES ────
tok_df = pd.read_csv("Data Inggris - american_precah_kata.csv")              # story_id | judul | sentence_id | word

# keep only rows where word is a real token
tok_df = tok_df.dropna(subset=["word"]).copy()
tok_df["word"] = tok_df["word"].astype(str)                 # ensure str type

sent_df = (
    tok_df.groupby(["storyID", "sentenceID"])["word"]
          .agg(" ".join)                                     # join tokens with space
          .reset_index(name="text")
)

sent_lookup = {(r.storyID, r.sentenceID): r.text for r in sent_df.itertuples()}

# ── 3. EXPAND alias_df (one row per sentence) ────────────────
rows = []
for r in alias_df.itertuples():
    for sid in r.sentenceID:
        rows.append({
            "story_id"   : r.storyID,
            "person"     : r.characterID,
            "aliases"    : r.AliasCharacters,
            "sentence_id": sid,
            "text"       : sent_lookup.get((r.storyID, sid), "")
        })

out = pd.DataFrame(rows)

# ── 4. SAVE ──────────────────────────────────────────────────
out.to_csv("alias_sentence_text.csv", index=False)
print("✅ Saved as 'alias_sentence_text.csv'")


✅ Saved as 'alias_sentence_text.csv'


In [11]:
import pandas as pd
import ast
import re
from collections import defaultdict, Counter

# ═══ 1. LOAD DATA ════════════════════════════════════════════════════════
alias_df = pd.read_csv("alias_sentence_text.csv")   # story_id | person | aliases | sentence_id | text
token_df = pd.read_csv("Data Inggris - american_precah_kata.csv")   # story_id | judul  | sentence_id | word

alias_df["aliases"] = alias_df["aliases"].apply(ast.literal_eval)
token_df  = token_df.dropna(subset=["word"]).copy()
token_df["word"] = token_df["word"].astype(str)

# ═══ 2. PREP TOKENS PER CERITA ══════════════════════════════════════════
story_tokens = defaultdict(list)        # {story_id: [token1, token2, ...]}
for r in token_df.itertuples():
    story_tokens[r.storyID].append(r.word.lower())

story_strings = {sid: " ".join(tok) for sid, tok in story_tokens.items()}

# ═══ 3. HITUNG mention_count PER TOKOH (story-level) ════════════════════
mention_dict = {}   # key = (story_id, person) -> total count

# groupby agar satu tokoh (story_id, person) diproses sekali
for (sid, person), sub in alias_df.groupby(["story_id", "person"]):
    # satukan semua alias dari baris-baris tokoh tsb
    alias_set = set()
    for alist in sub["aliases"]:
        alias_set.update([a.lower() for a in alist])

    toks = story_tokens[sid]
    txt  = story_strings[sid]

    count = 0
    for a in alias_set:
        if len(a.split()) == 1:                 # single-word alias
            count += toks.count(a)
        else:                                   # multi-word alias
            count += len(re.findall(r"\b" + re.escape(a) + r"\b", txt))

    mention_dict[(sid, person)] = count

# ═══ 4. TEMPEL KE TABEL alias_sentence_text ═════════════════════════════
alias_df["mention_count"] = alias_df.apply(
    lambda r: mention_dict[(r.story_id, r.person)], axis=1
)

# ═══ 5. (OPSI) URUTKAN & SIMPAN ═════════════════════════════════════════
alias_df = alias_df.sort_values(["story_id", "sentence_id", "person"])
alias_df.to_csv("alias_sentence_features_revised.csv", index=False)

print("✅  mention_count kini dihitung per tokoh & disimpan di 'alias_sentence_features_revised.csv'")


✅  mention_count kini dihitung per tokoh & disimpan di 'alias_sentence_features_revised.csv'


In [12]:
import pandas as pd
import ast

# ── 1. LOAD file hasil sebelumnya ─────────────────────────────
df = pd.read_csv("alias_sentence_features_revised.csv")   # story_id | person | aliases | sentence_id | text | mention_count
df["aliases"] = df["aliases"].apply(ast.literal_eval)

# ── 2. HITUNG panjang setiap kalimat (jumlah kata) ────────────
df["sent_len"] = df["text"].str.split().str.len()         # panjang per-kalimat

# ── 3. AGREGASI word_count per TOKOH (story-level) ───────────
total_wc = (                                              # {(story_id, person): total_words}
    df.groupby(["story_id", "person"])["sent_len"]
      .sum()
      .to_dict()
)

df["word_count"] = df.apply(
    lambda r: total_wc[(r.story_id, r.person)], axis=1
)

# ── 4. BERSIHkan kolom temp & simpan ─────────────────────────
df = df.drop(columns=["sent_len"])
df.to_csv("alias_sentence_features_wc.csv", index=False)

print("✅  Kolom word_count (total kata per tokoh di cerita) sudah ditambahkan dan disimpan ke 'alias_sentence_features_wc.csv'")


✅  Kolom word_count (total kata per tokoh di cerita) sudah ditambahkan dan disimpan ke 'alias_sentence_features_wc.csv'


In [13]:
import pandas as pd
import re

# ── 1. LOAD file ───────────────────────────────────────────────
df = pd.read_csv("alias_sentence_features_wc.csv")

# ── 2. EXTRACT NOMOR TOKOH (Tokoh-7 → 7) ───────────────────────
df["person_num"] = (
    df["person"]
      .str.extract(r"Person-(\d+)", expand=False)
      .astype(int)
)

# ── 3. SORT: story_id  → nomor tokoh  → sentence_id ────────────
df = (
    df.sort_values(["story_id", "person_num", "sentence_id"])
      .reset_index(drop=True)
)

# ── 4. DROP kolom bantu & SIMPAN ───────────────────────────────
df = df.drop(columns=["person_num"])
df.to_csv("alias_sentence_features_sorted.csv", index=False)

print("✅  File disimpan sebagai 'alias_sentence_features_sorted.csv' dengan urutan Tokoh-1, Tokoh-2, dst per cerita.")


✅  File disimpan sebagai 'alias_sentence_features_sorted.csv' dengan urutan Tokoh-1, Tokoh-2, dst per cerita.


In [17]:
import pandas as pd
import ast
import re

# ── 1.  LOAD dua file ────────────────────────────────────────────────
feat = pd.read_csv("alias_sentence_features_wc.csv")    # story_id | person | ... | text
feat["aliases"] = feat["aliases"].apply(ast.literal_eval)

tok  = pd.read_csv("Data Inggris - american_precah_kata.csv")           # story_id | judul | sentence_id | word
tok   = tok.dropna(subset=["word"]).copy()
tok["word"] = tok["word"].astype(str)

# ── 2.  SUSUN kalimat utuh & lookup───────────────────────────────────
sent_df = (
    tok.groupby(["storyID", "sentenceID"])["word"]
       .agg(" ".join)
       .reset_index(name="full_text")
)

lookup = {(r.storyID, r.sentenceID): r.full_text for r in sent_df.itertuples()}

def get_prev(row):
    return lookup.get((row.story_id, row.sentence_id - 1), "")

def get_next(row):
    return lookup.get((row.story_id, row.sentence_id + 1), "")

# ── 3.  TAMBAH kolom context ─────────────────────────────────────────
feat["text_prev"] = feat.apply(get_prev, axis=1)
feat["text_next"] = feat.apply(get_next, axis=1)

feat["bert_context"] = (
    feat["text_prev"].str.strip() + " [SEP] " +
    feat["text"].str.strip()      + " [SEP] " +
    feat["text_next"].str.strip()
).str.strip()

# ── 4.  SIMPAN ───────────────────────────────────────────────────────
feat.to_csv("alias_sentence_features_context.csv", index=False)
print("✅  text_prev, text_next, bert_context ditambahkan (pakai tokenized_sentences.csv) ➜ alias_sentence_features_context.csv")


✅  text_prev, text_next, bert_context ditambahkan (pakai tokenized_sentences.csv) ➜ alias_sentence_features_context.csv


In [18]:
import pandas as pd
import ast
import re

# ── 1. LOAD the current feature file (with text, aliases, etc.) ──
df = pd.read_csv("alias_sentence_features_context.csv")   # ganti nama jika berbeda
df["aliases"] = df["aliases"].apply(ast.literal_eval)      # list asli

# ── 2. Kumpulkan SEMUA alias, buat regex word-boundary ──────────
all_aliases = (
    df["aliases"]
      .explode()
      .dropna()
      .map(lambda x: x.lower().strip())
      .unique()
      .tolist()
)
alias_patterns = {a: re.compile(r"\b" + re.escape(a) + r"\b") for a in all_aliases}

# ── 3. Tentukan apakah tokoh ini alias PERTAMA di kalimat ───────
def is_primary(row):
    sent = str(row["text"]).lower()
    first_pos, first_alias = None, None
    for alias, pat in alias_patterns.items():
        m = pat.search(sent)
        if m:
            pos = m.start()
            if first_pos is None or pos < first_pos:
                first_pos, first_alias = pos, alias
    if first_alias is None:
        return 0
    return 1 if first_alias in [a.lower().strip() for a in row["aliases"]] else 0

df["is_primary_in_sentence"] = df.apply(is_primary, axis=1)

# ── 4. Sort: story_id → nomor Tokoh → sentence_id ───────────────
df["person_num"] = (
    df["person"]
      .str.extract(r"Person-(\d+)", expand=False)
      .astype(int)
)
df = (
    df.sort_values(["story_id", "person_num", "sentence_id"])
      .reset_index(drop=True)
      .drop(columns=["person_num"])
)

# ── 5. Simpan hasil ─────────────────────────────────────────────
df.to_csv("alias_sentence_features_primary_sorted.csv", index=False)
print("✅  is_primary_in_sentence diperbaiki & file di-sort. Hasil: 'alias_sentence_features_primary_sorted.csv'")


✅  is_primary_in_sentence diperbaiki & file di-sort. Hasil: 'alias_sentence_features_primary_sorted.csv'
